# 00 · EDA ràpida

Plantilla per a **qualsevol dataset nou** el primer dia. Objectiu: en 45 minuts saber la forma de les dades, el target, els nuls, el possible leakage i **5 insights** per a l'slide.

> Canvieu només la cel·la `CONFIG`. Ve configurada amb el repte Schneider 2025 (`bash scripts/download_past_data.sh`).

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from dreamteam import data

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

In [ ]:
# ── CONFIG ──────────────────────────────────────────────
PATH   = ROOT / "data/past/2025_schneider/dataset.csv"
TARGET = "target_variable"
ID_COLS = ["id"]          # columnes que NO són features
TIME_COL = None           # p. ex. "id_season" si hi ha temps
# ────────────────────────────────────────────────────────
df = data.load(PATH)
df = data.reduce_memory(df)
print(df.shape)
df.head()

## 1. Visió general: tipus, nuls i cardinalitat

In [ ]:
data.summary(df)

## 2. Target

Classificació: està balancejat? Regressió: és esbiaixat? Si ho és, cal `log1p` (vegeu `docs/04_playbook_ml.md`).

In [ ]:
y = df[TARGET]
if y.nunique() <= 20:
    print(y.value_counts(normalize=True).round(3))
    y.value_counts().plot.bar(title=f"Distribució de {TARGET}")
else:
    print(y.describe())
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    y.hist(bins=50, ax=ax[0]); ax[0].set_title(TARGET)
    np.log1p(y.clip(lower=0)).hist(bins=50, ax=ax[1]); ax[1].set_title(f"log1p({TARGET})")
plt.show()

## 3. Relació de cada variable amb el target

Busqueu **variables massa bones** (possible leakage) i relacions no lineals.

In [ ]:
features = [c for c in df.columns if c not in [TARGET, *ID_COLS]]
num = df[features].select_dtypes("number").columns
cat = [c for c in features if c not in num]

corr = df[[*num, TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print("Correlació amb el target (|r| > 0,9 → sospiteu leakage):")
print(corr.round(3).to_string())

In [ ]:
top = corr.index[:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, top):
    if df[TARGET].nunique() <= 20:
        sns.kdeplot(data=df, x=col, hue=TARGET, common_norm=False, ax=ax, warn_singular=False)
    else:
        sns.scatterplot(data=df.sample(min(len(df), 3000), random_state=0), x=col, y=TARGET, ax=ax, s=8)
    ax.set_title(col)
plt.tight_layout(); plt.show()

In [ ]:
# Categòriques: taxa o mitjana del target per categoria (les 15 més freqüents)
for col in cat[:6]:
    g = df.groupby(col, observed=True)[TARGET].agg(["mean", "count"]).sort_values("count", ascending=False).head(15)
    display(g.style.background_gradient(subset=["mean"]).set_caption(col))

## 4. Temps (si n'hi ha)

In [ ]:
if TIME_COL:
    df.groupby(TIME_COL)[TARGET].agg(["mean", "count"]).plot(subplots=True, figsize=(10, 5), marker="o")
    plt.show()
else:
    print("Sense columna temporal → probablement un K-Fold (estratificat) és la validació correcta.")

## 5. Insights per a la presentació

1. …
2. …
3. …
4. …
5. …

**Validació que farem servir:** … **perquè** …